# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RAJkumar5736602/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Heavy tails in traffic and length; medians far below means.

In [1]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p/'data/raw').exists())
sys.path.insert(0, str(root/'work/scripts'))
from wk import *
import pandas as pd, numpy as np
df = load()
print(df[['impressions_90d','clicks_90d','word_count','content_age_days','avg_position']].describe(percentiles=[.5,.9,.99]).round(1).T)

                    count    mean      std   min     50%      90%      99%  \
impressions_90d   30000.0  5200.4  16838.0   1.0   731.0  12136.4  73505.8   
clicks_90d        30000.0    16.1     75.1   0.0     1.0     32.0    253.0   
word_count        22301.0  3107.8   1452.4   8.0  2877.0   5327.0   7292.0   
content_age_days  30000.0   256.2    132.7  90.0   236.0    463.0    537.0   
avg_position      30000.0    16.3     15.2   0.0    10.8     36.8     69.9   

                       max  
impressions_90d   517715.0  
clicks_90d          4178.0  
word_count          9546.0  
content_age_days     564.0  
avg_position         245.0  


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Three signals vs. decline rate (quartile bins). Verdicts are written from the output.

In [2]:
for c in ['word_count','content_age_days','avg_position']:
    q = pd.qcut(pd.to_numeric(df[c],errors='coerce'), 4, duplicates='drop')
    print(df.groupby(q, observed=True).y.agg(['mean','size']).round(3), '\n')

                   mean  size
word_count                   
(7.999, 2413.0]   0.497  5576
(2413.0, 2877.0]  0.595  5586
(2877.0, 3666.0]  0.583  5566
(3666.0, 9546.0]  0.599  5573 

                   mean  size
content_age_days             
(89.999, 132.0]   0.600  7518
(132.0, 236.0]    0.640  8128
(236.0, 333.0]    0.494  6917
(333.0, 564.0]    0.422  7437 

                mean  size
avg_position              
(-0.001, 6.2]  0.461  7543
(6.2, 10.8]    0.580  7534
(10.8, 22.3]   0.611  7462
(22.3, 245.0]  0.517  7461 



## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Flag-linked test: the paper's 'expand thin pages' advice assumes shorter pages decline more. Compare decline rate for <2000 vs >=2000 words among pages with >=1000 impressions.

In [3]:
vis = df[df.impressions_90d>=1000]
print(vis.groupby(vis.word_count<2000).y.agg(['mean','size']).rename(index={True:'<2000 words',False:'>=2000 words'}).round(3))
print('competition level:'); print(df.groupby('competition_level').y.agg(['mean','size']).round(3))

               mean   size
word_count                
>=2000 words  0.583  12663
<2000 words   0.768    849
competition level:
                    mean   size
competition_level              
HIGH               0.556   2658
LOW                0.563  22896
MEDIUM             0.567   1836


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

See FINDINGS.md for the verdicts written from these numbers.

In [4]:
print('done')

done


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.